# SQL Complete Practice — Part 06: SQL Functions
**Topics 76–120 | MySQL | Ankanhaldar**

This notebook covers aggregate, string, numeric, date/time, and NULL-handling functions in MySQL. Every topic has its own theory section and SQL example cell.

> **Execution note:** These are MySQL SQL cells, not Python. Run them in MySQL Workbench, VS Code with SQLTools, or a compatible SQL notebook/kernel. A standard Python kernel does not execute SQL directly.

> **Practice database:** Run the setup cell first. The examples use employee and order sample tables.

In [ ]:
CREATE DATABASE IF NOT EXISTS ankan_sql_practice;
USE ankan_sql_practice;

CREATE TABLE IF NOT EXISTS function_demo_employee (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(80) NOT NULL,
    department VARCHAR(50),
    salary DECIMAL(10, 2),
    joining_date DATE,
    bonus DECIMAL(10, 2) NULL
);

INSERT INTO function_demo_employee
    (employee_id, employee_name, department, salary, joining_date, bonus)
VALUES
    (1, 'Amit Sen', 'IT', 45000, '2022-04-10', 5000),
    (2, 'Riya Das', 'HR', 38000, '2021-08-15', NULL),
    (3, 'Suman Roy', 'IT', 60000, '2020-01-20', 8000),
    (4, 'Priya Paul', 'Finance', 52000, '2023-03-05', 6000),
    (5, 'Rahul Dey', 'Sales', 35000, '2024-06-12', NULL),
    (6, 'Neha Bose', 'HR', 42000, '2022-11-01', 3000)
ON DUPLICATE KEY UPDATE
    employee_name = VALUES(employee_name),
    department = VALUES(department),
    salary = VALUES(salary),
    joining_date = VALUES(joining_date),
    bonus = VALUES(bonus);

CREATE TABLE IF NOT EXISTS function_demo_orders (
    order_id INT PRIMARY KEY,
    customer_name VARCHAR(80) NOT NULL,
    order_amount DECIMAL(10, 2),
    order_date DATETIME
);

INSERT INTO function_demo_orders
    (order_id, customer_name, order_amount, order_date)
VALUES
    (101, 'Amit', 1250.50, '2026-01-10 10:30:00'),
    (102, 'Riya', 2500.00, '2026-02-15 14:45:00'),
    (103, 'Suman', 799.99, '2026-02-20 09:15:00'),
    (104, 'Priya', 3200.75, '2026-03-05 18:20:00')
ON DUPLICATE KEY UPDATE
    customer_name = VALUES(customer_name),
    order_amount = VALUES(order_amount),
    order_date = VALUES(order_date);

SELECT * FROM function_demo_employee;
SELECT * FROM function_demo_orders;

**Setup note:** The sample inserts use `ON DUPLICATE KEY UPDATE`, so rerunning the setup refreshes existing sample records. Date/time examples use fixed sample dates to keep the output reproducible.

## Topic 76: What Are SQL Functions?

SQL functions are built-in operations that accept input values and return a result. They help calculate summaries, transform text, perform mathematical operations, work with dates and times, and handle missing values.

Common categories:
- Aggregate functions — calculate one result from multiple rows.
- String functions — manipulate text.
- Numeric functions — perform mathematical calculations.
- Date and time functions — extract or calculate date/time values.
- NULL-handling functions — work with missing or unknown values.

Functions can appear in `SELECT`, `WHERE`, `ORDER BY`, `HAVING`, and other expression contexts, subject to SQL rules.

### SQL Example — Topic 76

In [ ]:
SELECT
    UPPER(employee_name) AS uppercase_name,
    ROUND(salary * 1.10, 2) AS salary_with_10_percent,
    YEAR(joining_date) AS joining_year
FROM function_demo_employee;

**Expected result / explanation**

This query combines string, numeric, and date functions in one result. Each function transforms or calculates a value for each row.

## Topic 77: Aggregate Functions

Aggregate functions calculate a result from a set of rows. Common MySQL aggregate functions include `COUNT()`, `SUM()`, `AVG()`, `MIN()`, and `MAX()`.

They are often used with `GROUP BY` to calculate a separate result for each group. Without `GROUP BY`, an aggregate normally calculates one result for the entire filtered set.

### SQL Example — Topic 77

In [ ]:
SELECT
    COUNT(*) AS total_employees,
    SUM(salary) AS total_salary,
    AVG(salary) AS average_salary,
    MIN(salary) AS minimum_salary,
    MAX(salary) AS maximum_salary
FROM function_demo_employee;

**Expected result / explanation**

The query returns one summary row for the employee table. Each aggregate produces a different statistic.

## Topic 78: COUNT() Function

`COUNT()` counts rows or non-NULL values.

- `COUNT(*)` counts all rows.
- `COUNT(column_name)` counts rows where that column is not `NULL`.
- `COUNT(DISTINCT column_name)` counts distinct non-NULL values.

This distinction is important when a column contains missing values.

### SQL Example — Topic 78

In [ ]:
SELECT COUNT(*) AS all_employee_rows
FROM function_demo_employee;

SELECT COUNT(bonus) AS employees_with_bonus
FROM function_demo_employee;

SELECT COUNT(DISTINCT department) AS department_count
FROM function_demo_employee;

**Expected result / explanation**

The first query counts all employees. The second excludes employees whose bonus is `NULL`. The third counts unique non-NULL department values.

## Topic 79: SUM() Function

`SUM()` returns the total of numeric values in a column or expression. It ignores `NULL` values. If there are no non-NULL values to sum, the result is `NULL`.

It is commonly used for total sales, expenses, salaries, and quantities.

### SQL Example — Topic 79

In [ ]:
SELECT SUM(salary) AS total_salary
FROM function_demo_employee;

SELECT department, SUM(salary) AS department_salary_total
FROM function_demo_employee
GROUP BY department;

**Expected result / explanation**

The first query calculates the total salary across all employees. The second calculates a separate total for each department.

## Topic 80: AVG() Function

`AVG()` calculates the arithmetic average of non-NULL numeric values.

It is calculated as the sum of non-NULL values divided by the count of non-NULL values. It does not treat `NULL` as zero.

### SQL Example — Topic 80

In [ ]:
SELECT AVG(salary) AS average_salary
FROM function_demo_employee;

SELECT department, AVG(salary) AS department_average
FROM function_demo_employee
GROUP BY department;

**Expected result / explanation**

The first query returns the overall average salary. The second returns the average salary within each department.

## Topic 81: MIN() Function

`MIN()` returns the smallest value in a set. It can be used with numbers, dates, and comparable text values. It ignores `NULL` values.

### SQL Example — Topic 81

In [ ]:
SELECT MIN(salary) AS lowest_salary
FROM function_demo_employee;

SELECT MIN(joining_date) AS earliest_joining_date
FROM function_demo_employee;

**Expected result / explanation**

The first query finds the lowest salary. The second finds the earliest joining date.

## Topic 82: MAX() Function

`MAX()` returns the largest value in a set. It works with numeric, date/time, and comparable string values, and ignores `NULL` values.

### SQL Example — Topic 82

In [ ]:
SELECT MAX(salary) AS highest_salary
FROM function_demo_employee;

SELECT MAX(joining_date) AS latest_joining_date
FROM function_demo_employee;

**Expected result / explanation**

The first query finds the highest salary. The second finds the most recent joining date.

## Topic 83: GROUP_CONCAT() Function

`GROUP_CONCAT()` concatenates values from multiple rows into one string. It is a MySQL aggregate function and is often used with `GROUP BY`.

Useful options include:
- `DISTINCT` to remove repeated values.
- `ORDER BY` to control the concatenation order.
- `SEPARATOR` to choose the separator string.

The maximum output length is controlled by the `group_concat_max_len` system variable.

### SQL Example — Topic 83

In [ ]:
SELECT
    department,
    GROUP_CONCAT(employee_name ORDER BY employee_name SEPARATOR ', ') AS employees
FROM function_demo_employee
GROUP BY department;

**Expected result / explanation**

Each department appears once, with its employee names joined into a comma-separated string.

## Topic 84: UPPER() / UCASE()

`UPPER()` converts a string to uppercase. `UCASE()` is a synonym in MySQL.

It is useful for display formatting and case normalization. Case comparison behaviour is also affected by the column collation, so changing display case is not always necessary for case-insensitive searches.

### SQL Example — Topic 84

In [ ]:
SELECT
    employee_name,
    UPPER(employee_name) AS uppercase_name,
    UCASE(department) AS uppercase_department
FROM function_demo_employee;

**Expected result / explanation**

The returned expressions display names and departments in uppercase. The stored values are not modified.

## Topic 85: LOWER() / LCASE()

`LOWER()` converts a string to lowercase. `LCASE()` is a MySQL synonym.

It is commonly used to normalize text for display or comparison expressions. It does not update the original stored value unless used in an `UPDATE` statement.

### SQL Example — Topic 85

In [ ]:
SELECT
    employee_name,
    LOWER(employee_name) AS lowercase_name,
    LCASE(department) AS lowercase_department
FROM function_demo_employee;

**Expected result / explanation**

The query returns lowercase versions of the selected text values without changing table data.

## Topic 86: LENGTH() Function

`LENGTH()` returns the length of a string in bytes, not necessarily the number of characters. In a multibyte character set, one character may use multiple bytes.

Use `CHAR_LENGTH()` when you need the number of characters.

### SQL Example — Topic 86

In [ ]:
SELECT
    employee_name,
    LENGTH(employee_name) AS byte_length
FROM function_demo_employee;

**Expected result / explanation**

For the sample English names using a single-byte-compatible representation, byte length corresponds to the visible character count. This may differ for multibyte text.

## Topic 87: CHAR_LENGTH() Function

`CHAR_LENGTH()` returns the number of characters in a string. Unlike `LENGTH()`, it does not count the number of encoded bytes.

It is useful when validating text length in characters.

### SQL Example — Topic 87

In [ ]:
SELECT
    employee_name,
    CHAR_LENGTH(employee_name) AS character_count
FROM function_demo_employee;

**Expected result / explanation**

The result reports the character count for each employee name.

## Topic 88: CONCAT() Function

`CONCAT()` joins two or more strings into one string.

In MySQL, if any argument to `CONCAT()` is `NULL`, the result is `NULL`. Use `CONCAT_WS()` when a separator is needed and NULL arguments should be skipped.

### SQL Example — Topic 88

In [ ]:
SELECT
    CONCAT(employee_name, ' works in ', department) AS employee_details
FROM function_demo_employee;

**Expected result / explanation**

Each result combines an employee name with the department text. If any argument is NULL, MySQL's `CONCAT()` returns NULL for that row.

## Topic 89: CONCAT_WS() Function

`CONCAT_WS()` means concatenate with separator. The first argument is the separator; subsequent string arguments are joined using it.

Unlike `CONCAT()`, `CONCAT_WS()` skips `NULL` values after the separator argument. The separator itself should not be `NULL`.

### SQL Example — Topic 89

In [ ]:
SELECT
    CONCAT_WS(' | ', employee_name, department, joining_date) AS employee_summary
FROM function_demo_employee;

**Expected result / explanation**

The values are joined using ` | `. Any NULL value among the joined arguments is skipped.

## Topic 90: SUBSTRING() / SUBSTR()

`SUBSTRING()` extracts part of a string. `SUBSTR()` is a synonym in MySQL.

The common form is `SUBSTRING(string, start_position, length)`. String positions begin at 1. A negative start position counts from the end of the string.

### SQL Example — Topic 90

In [ ]:
SELECT
    employee_name,
    SUBSTRING(employee_name, 1, 4) AS first_four_characters,
    SUBSTR(employee_name, -3) AS last_three_characters
FROM function_demo_employee;

**Expected result / explanation**

The first expression extracts four characters from the beginning. The second extracts the last three characters.

## Topic 91: LEFT() Function

`LEFT(string, length)` returns the leftmost specified number of characters from a string.

### SQL Example — Topic 91

In [ ]:
SELECT
    employee_name,
    LEFT(employee_name, 4) AS first_four_characters
FROM function_demo_employee;

**Expected result / explanation**

The result contains the first four characters of each employee name.

## Topic 92: RIGHT() Function

`RIGHT(string, length)` returns the rightmost specified number of characters from a string.

### SQL Example — Topic 92

In [ ]:
SELECT
    employee_name,
    RIGHT(employee_name, 3) AS last_three_characters
FROM function_demo_employee;

**Expected result / explanation**

The result contains the final three characters of each employee name.

## Topic 93: TRIM() Function

`TRIM()` removes unwanted characters from the beginning and end of a string. By default, it removes spaces.

MySQL also supports `LTRIM()` for leading spaces and `RTRIM()` for trailing spaces. `TRIM()` can be used with `LEADING`, `TRAILING`, or `BOTH` and a specified removal string.

### SQL Example — Topic 93

In [ ]:
SELECT
    TRIM('   SQL Practice   ') AS trimmed_text,
    LTRIM('   SQL Practice') AS left_trimmed,
    RTRIM('SQL Practice   ') AS right_trimmed;

**Expected result / explanation**

The expressions remove surrounding, leading, or trailing spaces respectively.

## Topic 94: REPLACE() Function

`REPLACE(string, from_string, to_string)` replaces every occurrence of a substring with another substring.

It returns a transformed string and does not update the stored column unless used in an `UPDATE` statement.

### SQL Example — Topic 94

In [ ]:
SELECT
    employee_name,
    REPLACE(employee_name, ' ', '_') AS modified_name
FROM function_demo_employee;

**Expected result / explanation**

Spaces in each employee name are replaced with underscores in the query result.

## Topic 95: REVERSE() Function

`REVERSE()` returns a string with its character order reversed. It is useful for text transformation exercises and some string-processing tasks.

### SQL Example — Topic 95

In [ ]:
SELECT
    employee_name,
    REVERSE(employee_name) AS reversed_name
FROM function_demo_employee;

**Expected result / explanation**

Each employee name is displayed in reverse character order. The original data is unchanged.

## Topic 96: LOCATE() Function

`LOCATE(substring, string)` returns the position of the first occurrence of a substring in a string. Positions begin at 1. It returns 0 if the substring is not found.

An optional third argument specifies the position at which the search begins.

### SQL Example — Topic 96

In [ ]:
SELECT
    employee_name,
    LOCATE('a', employee_name) AS first_a_position
FROM function_demo_employee;

**Expected result / explanation**

The result gives the first position of lowercase `a` in each name, or 0 if it is absent. Matching behaviour can depend on collation.

## Topic 97: INSTR() Function

`INSTR(string, substring)` returns the position of the first occurrence of a substring. It is similar to `LOCATE()`, with the argument order reversed.

### SQL Example — Topic 97

In [ ]:
SELECT
    employee_name,
    INSTR(employee_name, 'a') AS a_position
FROM function_demo_employee;

**Expected result / explanation**

The query reports the first position of lowercase `a`, or 0 when it is not found.

## Topic 98: LPAD() Function

`LPAD(string, length, pad_string)` adds characters to the left of a string until it reaches the requested total length.

If the original string is longer than the requested length, MySQL truncates it to the requested length. Choose the target length carefully.

### SQL Example — Topic 98

In [ ]:
SELECT
    employee_id,
    LPAD(CAST(employee_id AS CHAR), 5, '0') AS formatted_id
FROM function_demo_employee;

**Expected result / explanation**

Numeric IDs are converted to text and padded on the left with zeroes, such as `00001`.

## Topic 99: RPAD() Function

`RPAD(string, length, pad_string)` adds characters to the right of a string until it reaches the requested total length.

If the original string is longer than the requested length, it is truncated.

### SQL Example — Topic 99

In [ ]:
SELECT
    department,
    RPAD(department, 12, '.') AS padded_department
FROM function_demo_employee;

**Expected result / explanation**

The department text is padded on the right with periods to a total length of 12 characters.

## Topic 100: FORMAT() Function

MySQL's `FORMAT(number, decimal_places)` formats a number with the specified decimal places and thousands separators. It returns a string, not a numeric value.

The optional third argument controls the locale. Do not use the formatted string for further numeric calculations without converting it back to a number.

### SQL Example — Topic 100

In [ ]:
SELECT
    salary,
    FORMAT(salary, 2) AS formatted_salary
FROM function_demo_employee;

**Expected result / explanation**

The salary is displayed with two decimal places and locale-style grouping separators.

## Topic 101: ROUND() Function

`ROUND(number, decimals)` rounds a number to the specified number of decimal places. If the decimal-place argument is omitted, it rounds to the nearest integer.

It is commonly used for reporting and financial presentation. Be aware that rounding a displayed result does not change the stored value.

### SQL Example — Topic 101

In [ ]:
SELECT
    order_amount,
    ROUND(order_amount) AS rounded_integer,
    ROUND(order_amount, 1) AS rounded_one_decimal
FROM function_demo_orders;

**Expected result / explanation**

The query displays each order amount rounded to an integer and to one decimal place.

## Topic 102: CEIL() / CEILING()

`CEIL()` and `CEILING()` return the smallest integer greater than or equal to a number. They are synonyms in MySQL.

For example, `CEIL(4.2)` returns 5, while `CEIL(4.0)` returns 4.

### SQL Example — Topic 102

In [ ]:
SELECT
    order_amount,
    CEIL(order_amount) AS ceiling_value,
    CEILING(order_amount) AS ceiling_alias
FROM function_demo_orders;

**Expected result / explanation**

Both functions return the same ceiling value for each order amount.

## Topic 103: FLOOR() Function

`FLOOR(number)` returns the largest integer less than or equal to a number.

For example, `FLOOR(4.9)` returns 4. For negative values, it rounds downward toward negative infinity.

### SQL Example — Topic 103

In [ ]:
SELECT
    order_amount,
    FLOOR(order_amount) AS floor_value
FROM function_demo_orders;

**Expected result / explanation**

The result contains the integer floor of each order amount.

## Topic 104: ABS() Function

`ABS(number)` returns the absolute value of a number, removing its negative sign if present.

It is useful when only the magnitude of a difference matters.

### SQL Example — Topic 104

In [ ]:
SELECT
    ABS(-125.75) AS absolute_value,
    ABS(125.75) AS positive_absolute_value;

**Expected result / explanation**

Both expressions return 125.75.

## Topic 105: MOD() Function

`MOD(number, divisor)` returns the remainder after division. The `%` operator is another way to perform modulus in MySQL.

A zero divisor does not produce a meaningful remainder; avoid dividing by zero.

### SQL Example — Topic 105

In [ ]:
SELECT
    MOD(17, 5) AS remainder_value,
    17 % 5 AS modulus_operator_result;

**Expected result / explanation**

Both expressions return 2 because 17 divided by 5 leaves a remainder of 2.

## Topic 106: POWER() / POW()

`POWER(base, exponent)` raises a number to a specified power. `POW()` is a synonym in MySQL.

### SQL Example — Topic 106

In [ ]:
SELECT
    POWER(2, 3) AS power_result,
    POW(5, 2) AS pow_result;

**Expected result / explanation**

The results are 8 and 25 respectively.

## Topic 107: SQRT() Function

`SQRT(number)` returns the square root of a non-negative number. A negative input does not produce a real-number square root.

### SQL Example — Topic 107

In [ ]:
SELECT
    SQRT(64) AS square_root,
    SQRT(2) AS square_root_of_two;

**Expected result / explanation**

The first result is 8. The second is an approximate decimal value.

## Topic 108: MOD Numeric Functions: GREATEST() and LEAST()

`GREATEST()` returns the largest value from its arguments, while `LEAST()` returns the smallest. Arguments should be comparable and compatible in type.

In MySQL, if any argument is `NULL`, these functions return `NULL`.

### SQL Example — Topic 108

In [ ]:
SELECT
    GREATEST(10, 25, 7, 18) AS greatest_value,
    LEAST(10, 25, 7, 18) AS least_value;

**Expected result / explanation**

The greatest value is 25 and the least value is 7.

## Topic 109: NOW() Function

`NOW()` returns the current date and time at the start of the current statement, in the session time zone.

It is useful for audit timestamps and time-based reporting. The exact result changes each time the query is executed.

### SQL Example — Topic 109

In [ ]:
SELECT NOW() AS current_datetime;

**Expected result / explanation**

The result is the current date and time according to the MySQL session time zone.

## Topic 110: CURDATE() / CURRENT_DATE()

`CURDATE()` returns the current date without a time component. `CURRENT_DATE()` is a synonym.

Use it when only the current calendar date is needed.

### SQL Example — Topic 110

In [ ]:
SELECT
    CURDATE() AS current_date_value,
    CURRENT_DATE() AS current_date_synonym;

**Expected result / explanation**

Both expressions return the current date for the MySQL session.

## Topic 111: CURTIME() / CURRENT_TIME()

`CURTIME()` returns the current time without the date. `CURRENT_TIME()` is a synonym.

The returned time follows the MySQL session time zone.

### SQL Example — Topic 111

In [ ]:
SELECT
    CURTIME() AS current_time_value,
    CURRENT_TIME() AS current_time_synonym;

**Expected result / explanation**

Both expressions return the current session time.

## Topic 112: DATE() Function

`DATE(datetime_expression)` extracts the date portion from a `DATE`, `DATETIME`, or compatible expression.

### SQL Example — Topic 112

In [ ]:
SELECT
    order_date,
    DATE(order_date) AS order_day
FROM function_demo_orders;

**Expected result / explanation**

The time portion is removed from each order timestamp in the returned expression.

## Topic 113: TIME() Function

`TIME(datetime_expression)` extracts the time portion from a `DATETIME` or compatible expression.

### SQL Example — Topic 113

In [ ]:
SELECT
    order_date,
    TIME(order_date) AS order_time
FROM function_demo_orders;

**Expected result / explanation**

The date portion is removed, leaving the time value.

## Topic 114: YEAR(), MONTH(), DAY()

These functions extract date components:
- `YEAR(date)` returns the year.
- `MONTH(date)` returns the month number (1–12).
- `DAY(date)` returns the day of the month (1–31).

They are useful for date-based grouping and filtering.

### SQL Example — Topic 114

In [ ]:
SELECT
    order_date,
    YEAR(order_date) AS order_year,
    MONTH(order_date) AS order_month,
    DAY(order_date) AS order_day
FROM function_demo_orders;

**Expected result / explanation**

The query extracts year, month, and day values from each order timestamp.

## Topic 115: HOUR(), MINUTE(), SECOND()

These functions extract time components from a time or datetime expression:
- `HOUR()` returns the hour.
- `MINUTE()` returns the minute.
- `SECOND()` returns the second.

### SQL Example — Topic 115

In [ ]:
SELECT
    order_date,
    HOUR(order_date) AS order_hour,
    MINUTE(order_date) AS order_minute,
    SECOND(order_date) AS order_second
FROM function_demo_orders;

**Expected result / explanation**

The result displays the hour, minute, and second components for each order timestamp.

## Topic 116: DATEDIFF() Function

`DATEDIFF(date1, date2)` returns the number of days between two dates, calculated as `date1 - date2`. Time portions are ignored.

The result can be positive, negative, or zero depending on the order of the dates.

### SQL Example — Topic 116

In [ ]:
SELECT
    order_id,
    DATEDIFF(DATE('2026-04-01'), DATE(order_date)) AS days_until_reference_date
FROM function_demo_orders;

**Expected result / explanation**

The query calculates the number of calendar days from each order date to 1 April 2026.

## Topic 117: DATE_ADD() / DATE_SUB()

`DATE_ADD()` adds an interval to a date or datetime. `DATE_SUB()` subtracts an interval.

Syntax examples:
- `DATE_ADD(date_value, INTERVAL 7 DAY)`
- `DATE_SUB(date_value, INTERVAL 1 MONTH)`

MySQL supports interval units such as DAY, MONTH, YEAR, HOUR, and MINUTE.

### SQL Example — Topic 117

In [ ]:
SELECT
    order_date,
    DATE_ADD(order_date, INTERVAL 7 DAY) AS after_seven_days,
    DATE_SUB(order_date, INTERVAL 1 MONTH) AS one_month_before
FROM function_demo_orders;

**Expected result / explanation**

The query calculates a date seven days later and a date one month earlier for each order timestamp.

## Topic 118: DATE_FORMAT() Function

`DATE_FORMAT(date, format)` converts a date or datetime value into a formatted string.

Common format specifiers:
- `%Y` — four-digit year.
- `%y` — two-digit year.
- `%m` — month number.
- `%M` — month name.
- `%d` — day of month.
- `%H` — hour (00–23).
- `%i` — minute.
- `%s` — second.

The result is text, not a date value.

### SQL Example — Topic 118

In [ ]:
SELECT
    order_date,
    DATE_FORMAT(order_date, '%d-%m-%Y') AS formatted_date,
    DATE_FORMAT(order_date, '%W, %M %d %Y') AS readable_date
FROM function_demo_orders;

**Expected result / explanation**

The first expression formats the date as day-month-year. The second produces a more readable weekday and month name.

## Topic 119: IFNULL() and COALESCE()

`IFNULL(expression, replacement)` returns the replacement when the expression is `NULL`; otherwise, it returns the expression.

`COALESCE(value1, value2, ...)` returns the first non-NULL argument. It is standard SQL and can test more than two alternatives.

These functions are useful for displaying defaults or handling missing values in calculations.

### SQL Example — Topic 119

In [ ]:
SELECT
    employee_name,
    bonus,
    IFNULL(bonus, 0) AS bonus_or_zero,
    COALESCE(bonus, salary, 0) AS first_available_value
FROM function_demo_employee;

**Expected result / explanation**

Employees without a bonus display 0 in `bonus_or_zero`. `COALESCE()` returns the bonus if present, otherwise salary, otherwise 0.

## Topic 120: NULLIF() Function

`NULLIF(expression1, expression2)` returns `NULL` when the two expressions are equal. Otherwise, it returns the first expression.

It is useful for preventing division by zero by converting a zero denominator to `NULL`. The resulting division then returns `NULL` rather than dividing by zero.

### SQL Example — Topic 120

In [ ]:
SELECT
    NULLIF(10, 10) AS equal_values_result,
    NULLIF(10, 5) AS different_values_result;

SELECT
    100 / NULLIF(0, 0) AS safe_division_result;

**Expected result / explanation**

The first query returns `NULL` for equal values and 10 for different values. The second returns `NULL` because the denominator becomes `NULL`.

## Quick Revision — Function Categories

| Category | Functions covered |
|---|---|
| Aggregate | `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`, `GROUP_CONCAT` |
| String | `UPPER`, `LOWER`, `LENGTH`, `CHAR_LENGTH`, `CONCAT`, `CONCAT_WS`, `SUBSTRING`, `LEFT`, `RIGHT`, `TRIM`, `REPLACE`, `REVERSE`, `LOCATE`, `INSTR`, `LPAD`, `RPAD`, `FORMAT` |
| Numeric | `ROUND`, `CEIL`, `CEILING`, `FLOOR`, `ABS`, `MOD`, `POWER`, `POW`, `SQRT`, `GREATEST`, `LEAST` |
| Date and time | `NOW`, `CURDATE`, `CURTIME`, `DATE`, `TIME`, `YEAR`, `MONTH`, `DAY`, `HOUR`, `MINUTE`, `SECOND`, `DATEDIFF`, `DATE_ADD`, `DATE_SUB`, `DATE_FORMAT` |
| NULL handling | `IFNULL`, `COALESCE`, `NULLIF` |

## Important Differences

### LENGTH() vs CHAR_LENGTH()
- `LENGTH()` counts bytes.
- `CHAR_LENGTH()` counts characters.

### CONCAT() vs CONCAT_WS()
- `CONCAT()` returns `NULL` if any argument is `NULL`.
- `CONCAT_WS()` skips `NULL` values after its separator argument.

### WHERE vs Aggregate Filtering
- Use `WHERE` to filter rows before grouping.
- Use `HAVING` to filter groups after aggregation.

### IFNULL() vs COALESCE()
- `IFNULL()` checks one expression and one replacement.
- `COALESCE()` returns the first non-NULL value from multiple expressions.

## Practice Checklist

- [ ] Calculate COUNT, SUM, AVG, MIN, and MAX.
- [ ] Group data and concatenate values using GROUP_CONCAT.
- [ ] Convert text using UPPER, LOWER, and CONCAT.
- [ ] Extract text using SUBSTRING, LEFT, and RIGHT.
- [ ] Clean text using TRIM and REPLACE.
- [ ] Apply ROUND, CEIL, FLOOR, ABS, and MOD.
- [ ] Extract date components and format dates.
- [ ] Calculate date differences and date intervals.
- [ ] Handle NULL values using IFNULL, COALESCE, and NULLIF.
- [ ] Explain LENGTH vs CHAR_LENGTH and CONCAT vs CONCAT_WS.

**End of Part 06 — Topics 76–120.**